**Financial Economics PS5 Problem 1 Part D**

This is the full 100,000-year IID simulation walkthrough for Problem 1(d). It contains the economic equations, hand-derived population moments, seven executable code cells, verified sample outputs, and the empirical interpretation. The source assignment PDF is titled Assignment 1; the shared repository is named financial-economics-ps5 as chosen for this submission.

Use ONE GitHub repository, financial-economics-ps5, for Problem 1(d) and the later computational work in Problem 2. Store this notebook at notebooks/problem1d.ipynb, its matching script at scripts/problem1d.py, and its outputs at results/problem1d/. Later, add notebooks/problem2b.ipynb and notebooks/problem2c.ipynb with their own scripts and results/problem2b/ and results/problem2c/ output directories. Those are future additions; today's runnable content is Problem 1(d).

Download and unpack the project ZIP. Create financial-economics-ps5 on GitHub, upload the unpacked CONTENTS at the repository root, and follow README.md for all setup and submission steps. GitHub stores and versions the files; Colab or GitHub Codespaces executes them.

For Colab, use File > Upload notebook to upload problem1d.ipynb from the notebooks folder. Save a personal copy in Drive and use a Python CPU runtime. Run the seven code cells in order with Shift+Enter or Runtime > Run all. The notebook is self-contained and does not require cloning the repository. In a standalone Colab runtime it saves results/problem1d/ under the current runtime directory, then creates problem1d_results.zip. Download both the executed notebook and the results ZIP. On GitHub, upload the executed notebook INSIDE notebooks/ and the extracted results/ folder at the repository root.

For Codespaces, open financial-economics-ps5 > Code > Codespaces > Create codespace on main. The bundled .devcontainer configuration sets up Python and its notebook kernel. Open notebooks/problem1d.ipynb, select the Python kernel and Run All. Alternatively run python scripts/problem1d.py from the repository root. The path helper finds the repository root even if the notebook starts in notebooks/, so outputs remain in results/problem1d/.

The supplied outputs use T=100,000 and seed=42. Rerun and save before submitting. Include https://github.com/YOUR_USERNAME/financial-economics-ps5 in the written submission, replacing YOUR_USERNAME with your actual account. A private repo requires the marker to have access. For a fixed notebook version, open it on GitHub and press y, then copy the permalink containing its commit SHA. The README explains how to preserve the submitted revision while adding Problem 2 later.


**1. Translate the economic model into four simulation equations.** Each observation is one annual transition, from time t to t+1. The dataframe row labels are observation numbers; the array delta_c contains growth, not the consumption level.

The common innovation is $\varepsilon_{t+1}\sim iid\,N(0,1)$. The annual calibration is $\sigma=.036,\ \mu=.018,\ \gamma=10,\ \beta=.99$.

$$
r^f=-\log\beta+\gamma\mu-\tfrac12\gamma^2\sigma^2,
\qquad \Delta c_{t+1}=\mu+\sigma\varepsilon_{t+1},
$$
$$
m_{t+1}=\log M_{t+1}=\log\beta-\gamma\Delta c_{t+1},
$$
$$
r^d_{t+1}=-\log\beta+\gamma\mu-\tfrac12(1-\gamma)^2\sigma^2+\sigma\varepsilon_{t+1}.
$$

Returns here are log gross returns, $r=\log(1+R)$, with R the simple net return in Kung's notation. The SDF series requested in the assignment is m, not M. Use the SAME innovation in every equation for a given year. Separate draws would destroy the model's economic relationships. Simulate these log quantities directly; accumulating 100,000 years of consumption levels and exponentiating them is unnecessary and can overflow. There is no burn-in: the model is IID from the first observation.

**2. Derive the population statistics before running the simulation.** For $X=a+b\varepsilon$, $E[X]=a$ and $\operatorname{Var}(X)=b^2$. Therefore:

| Series | Population mean | Population SD | Population ACF1 |
|---|---:|---:|---:|
| Log riskfree rate r_f | 0.1252503359 | 0 | Undefined |
| Log consumption growth delta_c | 0.018 | 0.036 | 0 |
| Log SDF m | -0.1900503359 | 0.360 | 0 |
| Log equity return r_d | 0.1375623359 | 0.036 | 0 |

In particular, $E[m]=\log\beta-\gamma\mu$ and $SD(m)=\gamma\sigma$. The other stochastic series have innovation loading $\sigma$, hence SD $\sigma$. Independence of shocks across years gives zero lag-one covariance for each nonconstant series. Correlation for the constant riskfree rate has a zero denominator and is undefined.

The common shock implies
$$
\operatorname{Cov}(\Delta c,r^d)=\sigma^2,
\qquad \operatorname{Corr}(\Delta c,r^d)=\frac{\sigma^2}{\sigma\sigma}=1.
$$
This correlation is exact in the model, including a finite simulated sample up to rounding. It is not merely a limit approached as T increases. Similarly, correlations of m with delta_c and r_d are -1. Means are included as an extra check; Part D specifically requests standard deviations, first autocorrelations, and the consumption/equity correlation.


**3. Set the calibration and import the libraries.** NumPy creates numerical arrays and random draws; pandas organizes results. T=100_000 means 100,000 annual observations, not 100,000 paths. The seed fixes a reproducible pseudo-random stream; 42 is a reproducibility choice, not an economic parameter. np.log is the natural logarithm; ** is exponentiation. Keep sigma=.036 and mu=.018 in decimal units. SAVE_PATHS=False avoids saving the large raw dataset while preserving all requested statistics.

Colab normally supplies NumPy and pandas. If an import is missing, run `%pip install numpy pandas` in a separate code cell, restart the runtime if requested, then run from the top. The bundle's requirements.txt pins the versions used for verification. Codespaces uses Python 3.12 through the supplied development-container configuration.
The file-management helper finds the shared repository root and writes this problem's outputs to results/problem1d/. This keeps the same code usable from a repository-root terminal, the notebooks/ folder, or a standalone Colab runtime. Future Problem 2 notebooks will use their own output subfolders. The helper does not affect the economic simulation.


In [1]:
# Cell 1: imports, annual calibration, and reproducibility settings.
import json
import platform
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd

T = 100_000
SEED = 42
sigma = 0.036
mu = 0.018
gamma = 10.0
beta = 0.99
SAVE_PATHS = False  # True also exports all 100,000 simulated observations.
def find_project_root():
    # Scripts know their own location; notebooks may start in notebooks/.
    if "__file__" in globals():
        script = Path(__file__).resolve()
        if script.parent.name == "scripts":
            return script.parent.parent
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (
            (candidate / "notebooks").is_dir()
            and (candidate / "scripts").is_dir()
            and (candidate / "requirements.txt").is_file()
        ):
            return candidate
    # Standalone Colab: use its current runtime directory.
    return current

PROJECT_ROOT = find_project_root()
OUT_DIR = PROJECT_ROOT / "results" / "problem1d"

assert T >= 3 and sigma > 0 and gamma > 0 and 0 < beta < 1
print(f"Python {platform.python_version()}; NumPy {np.__version__}; pandas {pd.__version__}")
print(f"Annual observations: {T:,}; seed: {SEED}")


Python 3.12.14; NumPy 2.3.5; pandas 2.2.3
Annual observations: 100,000; seed: 42


**4. Simulate the economic series.** The PCG64 generator is initialized once and produces one vector of T shocks. We reuse epsilon rather than generating shocks inside each equation. np.full repeats the constant riskfree rate T times. The DataFrame has T rows and four named columns. The first five rows let you inspect the construction without printing 100,000 observations. Re-running this cell resets the generator to the same seed and reproduces the same data.


In [2]:
# Cell 2: ONE vector of shocks drives all four log series.
rng = np.random.Generator(np.random.PCG64(SEED))
epsilon = rng.standard_normal(T)

r_f_value = -np.log(beta) + gamma * mu - 0.5 * gamma**2 * sigma**2
delta_c = mu + sigma * epsilon
m = np.log(beta) - gamma * delta_c
r_d = (
    -np.log(beta) + gamma * mu
    - 0.5 * (1 - gamma)**2 * sigma**2
    + sigma * epsilon
)
r_f = np.full(T, r_f_value)

data = pd.DataFrame({"r_f": r_f, "delta_c": delta_c, "m": m, "r_d": r_d})
assert np.isfinite(data.to_numpy()).all()
print(data.head().to_string(index=False, float_format=lambda x: f"{x:.8f}"))


       r_f     delta_c           m        r_d
0.12525034  0.02896981 -0.29974848 0.14853215
0.12525034 -0.01943943  0.18434394 0.10012291
0.12525034  0.04501624 -0.46021277 0.16457858
0.12525034  0.05186033 -0.52865363 0.17142267
0.12525034 -0.05223727  0.51232233 0.06732507


**5. Define the requested estimators.** The sample mean is $\bar x=T^{-1}\sum_{t=1}^T x_t$, and the sample SD uses
$$
s_x=\sqrt{\frac{1}{T-1}\sum_{t=1}^T(x_t-\bar x)^2}.
$$
Thus np.std(..., ddof=1) divides by T-1. With T=100,000, choosing T instead would make a negligible numerical difference, but state your convention.

We use the conventional sample autocorrelation
$$
\widehat\rho_1=\frac{\sum_{t=2}^T(x_t-\bar x)(x_{t-1}-\bar x)}{\sum_{t=1}^T(x_t-\bar x)^2}.
$$
The slices z[1:] and z[:-1] pair each observation with its predecessor. Their dot product is the numerator; dot(z,z) is the denominator. This definition differs slightly in finite samples from pandas.Series.autocorr(), which computes a Pearson correlation of two shifted vectors with separate means. Both are defensible if stated; use one definition consistently.

Detect an exactly constant series BEFORE demeaning. Floating-point averaging of a constant array can leave a tiny residual, which can otherwise create a false autocorrelation close to one. The code returns zero SD and np.nan (undefined) ACF for r_f. N/A in printed results is the same undefined value.


In [3]:
# Cell 3: explicit sample statistics, with a constant-series check.
def sample_std(x):
    x = np.asarray(x, dtype=float)
    if np.all(x == x[0]):
        return 0.0
    return float(np.std(x, ddof=1))

def acf1(x):
    x = np.asarray(x, dtype=float)
    if np.all(x == x[0]):
        return np.nan  # Correlation is undefined for a constant series.
    z = x - np.mean(x)
    return float(np.dot(z[1:], z[:-1]) / np.dot(z, z))

def sample_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if np.all(x == x[0]) or np.all(y == y[0]):
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])

print("Sample SD uses ddof=1; ACF uses the full-sample mean.")
print("Constant series: SD = 0, ACF = N/A.")


Sample SD uses ddof=1; ACF uses the full-sample mean.
Constant series: SD = 0, ACF = N/A.


**6. Compute the results and compare with the analytical benchmarks.** The dictionary theory stores each population mean, SD and ACF. The loop computes the matching sample statistics for each DataFrame column. sample_std and sample_acf1 answer the principal numerical requests. Values are in decimal natural-log units: .036 is 3.6 percentage points of annual log growth; .36 is a standard deviation of the log SDF, not an ordinary investment return. The riskfree rate's ACF is N/A.


In [4]:
# Cell 4: population benchmarks derived by hand.
theory = {
    "r_f": (r_f_value, 0.0, np.nan),
    "delta_c": (mu, sigma, 0.0),
    "m": (np.log(beta) - gamma * mu, gamma * sigma, 0.0),
    "r_d": (
        -np.log(beta) + gamma * mu - 0.5 * (1 - gamma)**2 * sigma**2,
        sigma, 0.0,
    ),
}

rows = []
for name in data.columns:
    x = data[name].to_numpy()
    mean_theory, sd_theory, acf_theory = theory[name]
    rows.append({
        "series": name,
        "sample_mean": float(np.mean(x)),
        "theory_mean": mean_theory,
        "sample_std": sample_std(x),
        "theory_std": sd_theory,
        "sample_acf1": acf1(x),
        "theory_acf1": acf_theory,
    })
summary = pd.DataFrame(rows)
print(summary.to_string(index=False, float_format=lambda x: f"{x:.8f}", na_rep="N/A"))


 series  sample_mean  theory_mean  sample_std  theory_std  sample_acf1  theory_acf1
    r_f   0.12525034   0.12525034  0.00000000  0.00000000          N/A          N/A
delta_c   0.01784761   0.01800000  0.03613350  0.03600000   0.00129805   0.00000000
      m  -0.18852648  -0.19005034  0.36133500  0.36000000   0.00129805   0.00000000
    r_d   0.13740995   0.13756234  0.03613350  0.03600000   0.00129805   0.00000000


**7. Compute the required correlation.** The question asks for the contemporaneous correlation between log consumption growth and log equity return, not between their levels or between different years. The function uses NumPy's Pearson sample correlation. The optional full matrix also exposes the two -1 correlations implied by the shared innovation. Entries involving the constant riskfree rate are undefined.


In [5]:
# Cell 5: the required contemporaneous correlation, plus optional checks.
corr_c_equity = sample_corr(delta_c, r_d)
correlations = pd.DataFrame(np.nan, index=data.columns, columns=data.columns)
for name_x in data.columns:
    for name_y in data.columns:
        correlations.loc[name_x, name_y] = sample_corr(data[name_x], data[name_y])
print(f"Corr(log consumption growth, log equity return) = {corr_c_equity:.8f}")
print(correlations.to_string(float_format=lambda x: f"{x:.8f}", na_rep="N/A"))


Corr(log consumption growth, log equity return) = 1.00000000
         r_f     delta_c           m         r_d
r_f      N/A         N/A         N/A         N/A
delta_c  N/A  1.00000000 -1.00000000  1.00000000
m        N/A -1.00000000  1.00000000 -1.00000000
r_d      N/A  1.00000000 -1.00000000  1.00000000


**8. Verify the simulation against economic identities.** From part (b), define
$$
\log q=\log\beta+(1-\gamma)\mu+\tfrac12(1-\gamma)^2\sigma^2.
$$
Finite fundamental equity prices require q<1, and the exact realized return satisfies $r^d=\Delta c-\log q$. The assertions check this relationship and the shared-shock volatility/correlation implications. These checks help detect a sign error or independent shocks.

Euler pricing additionally requires $E[e^{m+r^f}]=E[e^{m+r^d}]=1$. Sample averages need not be exactly one. We report a Monte Carlo standard error and a z-score to quantify the discrepancy. For IID observations, a sample mean has standard error approximately SD/sqrt(T). The rough white-noise ACF reference interval is $\pm1.96/\sqrt T\approx\pm.0062$; this is a diagnostic, not a guaranteed bound.

The three stochastic series have identical sample ACFs up to rounding because each is an affine transformation of the same shock vector. Small nonzero sample ACFs reflect finite-sample variation, not persistent shocks in this model. Larger T reduces sampling error; it does not change the population implications.

The optional premium check uses $E[r^d-r^f]=(\gamma-\tfrac12)\sigma^2$. The lecture's $\gamma\sigma^2$ corresponds to $\log E[1+R^d]-r^f$, a different quantity. That convention does not alter any of the Part D volatility or correlation predictions.


In [6]:
# Cell 6: check pricing identities and distinguish simulation error from bugs.
log_q = np.log(beta) + (1 - gamma) * mu + 0.5 * (1 - gamma)**2 * sigma**2
assert log_q < 0, "The fundamental price-dividend ratio requires q < 1."
assert np.allclose(r_d, delta_c - log_q, rtol=0, atol=1e-12)
assert np.isclose(corr_c_equity, 1.0, rtol=0, atol=1e-12)
assert np.isclose(sample_std(m), gamma * sample_std(delta_c), rtol=0, atol=1e-12)
assert np.isclose(sample_std(r_d), sample_std(delta_c), rtol=0, atol=1e-12)
assert np.isnan(acf1(r_f))

euler_rows = []
for label, log_gross_return in [("riskfree", r_f), ("equity", r_d)]:
    pricing_product = np.exp(m + log_gross_return)
    estimate = float(np.mean(pricing_product))
    mc_se = sample_std(pricing_product) / np.sqrt(T)
    euler_rows.append({
        "asset": label,
        "sample_mean_M_times_gross_return": estimate,
        "population_value": 1.0,
        "monte_carlo_standard_error": mc_se,
        "z_score_from_one": (estimate - 1.0) / mc_se if mc_se > 0 else np.nan,
    })
euler_checks = pd.DataFrame(euler_rows)
print(euler_checks.to_string(index=False, float_format=lambda x: f"{x:.8f}"))
print(f"Approximate white-noise ACF reference band: +/- {1.96 / np.sqrt(T):.6f}")
print(f"Sample mean log excess return: {np.mean(r_d - r_f):.8f}")
print(f"Population mean log excess return: {(gamma - 0.5) * sigma**2:.8f}")
simple_equity_sd_theory = (
    np.exp(theory["r_d"][0] + 0.5 * sigma**2)
    * np.sqrt(np.expm1(sigma**2))
)
simple_equity_sd_sample = sample_std(np.expm1(r_d))
print(f"Supplementary simple equity-return SD: sample {simple_equity_sd_sample:.8f}; theory {simple_equity_sd_theory:.8f}")
print("Structural identities passed. Euler averages need only be close to one.")


   asset  sample_mean_M_times_gross_return  population_value  monte_carlo_standard_error  z_score_from_one
riskfree                        1.00196887        1.00000000                  0.00118275        1.66465753
  equity                        1.00173381        1.00000000                  0.00105750        1.63953737
Approximate white-noise ACF reference band: +/- 0.006198
Sample mean log excess return: 0.01215961
Population mean log excess return: 0.01231200
Supplementary simple equity-return SD: sample 0.04150199; theory 0.04134921
Structural identities passed. Euler averages need only be close to one.


**9. Export the evidence for your submission.** The next cell writes summary.csv, correlations.csv, euler_checks.csv, run_metadata.json and submission_paragraph.txt into results/problem1d/. It also creates problem1d_results.zip containing those outputs. The CSV files use N/A for undefined statistics. SAVE_PATHS=True additionally exports all simulated observations, although the assignment does not require a raw-data attachment.

In Colab, use the Files sidebar to download problem1d_results.zip. Alternatively, run the following two lines manually in an extra cell: `from google.colab import files` followed by `files.download(str(PROJECT_ROOT / "problem1d_results.zip"))`. Download the completed notebook as an .ipynb from its File menu as well. Saving a notebook preserves its cell outputs, but does not permanently preserve arbitrary files on Colab's temporary virtual machine. Upload the notebook and the extracted results/problem1d/ files to GitHub after running. Do not upload a ZIP and expect GitHub to extract it automatically.


In [7]:
# Cell 7: save the results, run metadata, and a draft submission paragraph.
OUT_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT_DIR / "summary.csv", index=False, float_format="%.12g", na_rep="N/A")
correlations.to_csv(OUT_DIR / "correlations.csv", index_label="series", float_format="%.12g", na_rep="N/A")
euler_checks.to_csv(OUT_DIR / "euler_checks.csv", index=False, float_format="%.12g")

metadata = {
    "repository": "financial-economics-ps5",
    "problem": "Assignment 1 Problem 1(d)",
    "frequency": "annual", "T": T, "seed": SEED,
    "random_generator": "NumPy Generator with PCG64",
    "parameters": {"sigma": sigma, "mu": mu, "gamma": gamma, "beta": beta},
    "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__},
    "units": "Natural-log units; growth and returns are annual; m is log SDF.",
    "sample_std": "ddof=1; exactly constant arrays return zero",
    "acf1": "sum((x[1:]-mean(x))*(x[:-1]-mean(x)))/sum((x-mean(x))**2)",
    "N/A": "Undefined because one or both variables have zero variance.",
    "burn_in": "None: all draws are IID and stationary from the first observation.",
    "saved_full_paths": SAVE_PATHS,
}
(OUT_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")

sample = summary.set_index("series")
submission = (
    f"I simulated {T:,} annual observations using the stated calibration and "
    f"one shared IID standard-normal shock per year (PCG64, seed {SEED}). "
    f"The sample standard deviations of log consumption growth, the log SDF, "
    f"and the log equity return were {sample.loc['delta_c', 'sample_std']:.6f}, "
    f"{sample.loc['m', 'sample_std']:.6f}, and {sample.loc['r_d', 'sample_std']:.6f}, "
    f"respectively, close to their population values of {sigma:.3f}, "
    f"{gamma * sigma:.3f}, and {sigma:.3f}. Their lag-one sample autocorrelations "
    f"were all approximately {sample.loc['delta_c', 'sample_acf1']:.6f}, "
    f"consistent with zero population serial correlation. The riskfree rate was "
    f"constant, with zero standard deviation and undefined autocorrelation. "
    f"The consumption-growth/equity-return correlation was {corr_c_equity:.6f}, "
    "as implied by the model's constant price-dividend ratio. The results match "
    "the model's theoretical moments but fail to match important empirical "
    "features discussed in the course: consumption growth is only moderately "
    "correlated with stock returns (Lecture 4, printed p. 146), and riskfree "
    "rates exhibit nonzero volatility (Lecture 4, printed p. 156). The model's "
    f"ordinary equity-return SD is {100 * simple_equity_sd_theory:.2f}%, versus "
    "16.54% for annual real S&P returns in 1889-1978 (Mehra and Prescott, 1985, "
    "Table 1). The model's "
    "3.6% consumption-growth volatility is a calibration input. This exercise "
    "does not estimate new moments from an empirical dataset."
)
(OUT_DIR / "submission_paragraph.txt").write_text(submission + "\n", encoding="utf-8")
if SAVE_PATHS:
    data.assign(epsilon=epsilon).to_csv(OUT_DIR / "simulated_paths.csv", index=False)

archive = PROJECT_ROOT / "problem1d_results.zip"
output_files = [
    OUT_DIR / "summary.csv", OUT_DIR / "correlations.csv",
    OUT_DIR / "euler_checks.csv", OUT_DIR / "run_metadata.json",
    OUT_DIR / "submission_paragraph.txt",
]
if SAVE_PATHS:
    output_files.append(OUT_DIR / "simulated_paths.csv")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for file in output_files:
        z.write(file, arcname=str(Path("results") / "problem1d" / file.name))
print(submission)
print("\nSaved:")
for file in output_files:
    print(f"  {file.relative_to(PROJECT_ROOT)}")
print(f"  {archive.relative_to(PROJECT_ROOT)}")


I simulated 100,000 annual observations using the stated calibration and one shared IID standard-normal shock per year (PCG64, seed 42). The sample standard deviations of log consumption growth, the log SDF, and the log equity return were 0.036133, 0.361335, and 0.036133, respectively, close to their population values of 0.036, 0.360, and 0.036. Their lag-one sample autocorrelations were all approximately 0.001298, consistent with zero population serial correlation. The riskfree rate was constant, with zero standard deviation and undefined autocorrelation. The consumption-growth/equity-return correlation was 1.000000, as implied by the model's constant price-dividend ratio. The results match the model's theoretical moments but fail to match important empirical features discussed in the course: consumption growth is only moderately correlated with stock returns (Lecture 4, printed p. 146), and riskfree rates exhibit nonzero volatility (Lecture 4, printed p. 156). The model's ordinary eq

**10. Interpret the results and answer the data question.** First separate two questions: did the simulation implement the model correctly, and does the model reproduce observed economic data? The first is assessed using the analytical moments above. The second requires comparison with evidence.

The model implies consumption-growth and equity log-return volatility of .036 and a log-SDF volatility of .36. The consumption volatility was chosen as a calibration input, so reproducing it in the simulation is a consistency check. Perfect consumption/equity correlation follows from C=D and the constant price-dividend ratio. Lecture 4, printed p.146, describes the empirical correlation as only moderate. Thus the model's correlation of one is too strong. The riskfree rate is constant in this IID model, while Lecture 4, printed p.156, discusses nonzero empirical riskfree-rate volatility (with the benchmark depending on the treatment of inflation risk). Zero equity-return serial correlation can be consistent with weak return predictability, but does not validate the full model. No new empirical dataset has been supplied or estimated in this exercise, so do not invent a sample period or claim new empirical estimates.

As a supplementary historical comparison, [Mehra and Prescott (1985), Table 1, printed p.147](https://rajnishmehra.com/wp-content/uploads/2023/09/The-Equity-Premium-A-Puzzle.pdf) reports a 16.54% SD of annual real S&P simple returns for 1889-1978. Convert our model to the same simple-return measure before comparing:
$$
SD(R^d)=e^{E[r^d]+\sigma^2/2}\sqrt{e^{\sigma^2}-1}\approx .0413492.
$$
Its 4.13% simple-return volatility is substantially lower. The 16.54% historical figure is not a log-return SD. The code prints both the model's analytical simple-return SD and its simulated counterpart.

Slide 59 of Kung's deck also gives the familiar low-riskfree-rate/high-equity-premium empirical benchmarks. They provide context for the model's puzzles, but the statistics specifically requested in Part D are SD, ACF1 and contemporaneous correlation. Means in our table and the optional premium calculation are supplementary.

Use the draft paragraph printed above as a starting point, and include the summary table in your written submission. Mention annual frequency, T=100,000, the calibration, seed=42, the SD/ACF definitions, and that riskfree ACF is undefined. A repo link supplements the submitted numerical table and economic interpretation; a private-repository link requires access permission for another reader.

**11. Common problems and their fixes.** NameError usually means an earlier cell was skipped; restart and run from the top. ModuleNotFoundError means a library is missing; install NumPy/pandas in the active interpreter. A riskfree ACF near one indicates a constant-series handling error. A consumption/equity correlation far from one usually indicates separate shocks, a wrong formula, or misaligned years. An SDF mean near -0.19 is correct for log M; M itself must be positive and is obtained with exp(m). A small nonzero sample ACF is expected. Changing the seed changes sample statistics slightly but cannot change the structural correlation of one. Avoid manually editing result CSVs to force agreement with theoretical moments; fix the code and rerun instead.

**12. Course and setup references.** Assignment 1, Problem 1(d), pp.1-2. Kung, Consumption-based asset pricing part 1, slides 5, 49 and 57-62. Gomes, L4, printed pp.143, 146 and 156 (physical PDF pp.4, 7 and 17). The original course PDFs are not included in this repository.

Official setup documentation: [Create a GitHub repository](https://docs.github.com/en/repositories/creating-and-managing-repositories/creating-a-new-repository), [upload files](https://docs.github.com/en/repositories/working-with-files/managing-files/adding-a-file-to-a-repository), [Colab FAQ](https://research.google.com/colaboratory/faq.html), [create a Codespace](https://docs.github.com/en/codespaces/developing-in-a-codespace/creating-a-codespace-for-a-repository), [notebook controls and kernels](https://docs.github.com/en/codespaces/developing-in-a-codespace/getting-started-with-github-codespaces-for-machine-learning), [commit and push](https://docs.github.com/en/codespaces/developing-in-a-codespace/using-source-control-in-your-codespace), [stop a Codespace](https://docs.github.com/en/codespaces/developing-in-a-codespace/stopping-and-starting-a-codespace).


**13. Use this same repository for Problem 2 and your submission.** Keep the current notebook and results intact as the IID benchmark. When we reach Problem 2(b), add notebooks/problem2b.ipynb and scripts/problem2b.py, writing results/problem2b/. For bond yields in Problem 2(c), use notebooks/problem2c.ipynb, scripts/problem2c.py and results/problem2c/. Update the root README index when those files exist. Reuse the environment and statistical definitions, but derive the new AR(1) and pricing equations before coding. Problem 1's IID assumption and lack of burn-in do not automatically carry over to an AR(1) simulation.

Before submission, restart the notebook kernel, run all cells, save the notebook, commit the named code and results files, and push to GitHub. Use the root README as the marker's index. Include your repository URL and optionally the final commit SHA in the written problem set. You can obtain a file permalink by viewing the notebook on GitHub and pressing y outside a text field. The permalink fixes the notebook version while later commits add work on Problem 2.

GitHub collaborator access to a private personal repository includes write access; follow the course's sharing instructions. If invited, the intended marker must accept before access is granted. A URL or permalink preserves the repository's normal access requirements.

Official references: [invite a collaborator](https://docs.github.com/en/repositories/managing-your-repositorys-settings-and-features/repository-access-and-collaboration/inviting-collaborators-to-a-personal-repository), [personal repository permissions](https://docs.github.com/en/repositories/managing-your-repositorys-settings-and-features/repository-access-and-collaboration/permission-levels-for-a-personal-account-repository), [permanent file links](https://docs.github.com/en/repositories/working-with-files/using-files/getting-permanent-links-to-files).
